# 🏥 딥러닝 기초 CNN/전이학습 실습과제

## 과제 개요
이 과제에서는 의료 이미지 데이터셋을 활용하여 CNN과 전이학습을 적용합니다.

### 사용 데이터셋 (선택 가능)
- **MedMNIST** (권장): 의료 이미지 벤치마크 데이터셋
  - PathMNIST: 대장 조직 병리 이미지 (9 클래스)
  - DermaMNIST: 피부 병변 이미지 (7 클래스)
  - BloodMNIST: 혈액 세포 이미지 (8 클래스)
  - BreastMNIST: 유방 초음파 이미지 (2 클래스)
- **CIFAR-10**: 일반 이미지 데이터셋 (빠른 테스트용)
- **Chest X-Ray**: 폐렴 X-Ray 데이터셋 (Kaggle 다운로드 필요)

### 학습 목표
1. CNN 모델 설계 및 구현 (Day 4)
2. 데이터 증강(Augmentation) 적용
3. 전이학습(Transfer Learning) 적용 (Day 5)
4. Feature Extraction vs Fine-tuning 비교
5. 성능 비교 리포트 작성

### 제출물
- 완성된 노트북 파일 (.ipynb)
- 각 실험의 결과 분석 및 인사이트

---

> 📝 **작성자**: 조태희 · OZ 코딩 스쿨 AI 헬스케어 부트캠프 6기
>
> ▶️ **실행 방법 (Colab)**
> 1. `런타임 → 런타임 유형 변경 → T4 GPU`를 선택합니다. (CPU로는 224×224 이미지 학습이 매우 오래 걸립니다.)
> 2. `런타임 → 모두 실행`. 처음 실행할 때 MedMNIST 데이터(PathMNIST)와 ImageNet 사전학습 가중치(ResNet18, EfficientNet-B0)를 인터넷에서 자동으로 내려받습니다.
> 3. ⏱️ 아래 **`QUICK_MODE=True`**(기본값)는 학습 20,000장 / 검증 4,000장만 사용해 4개 모델의 학습 시간을 크게 줄여 줍니다(테스트는 전체 사용). 시간이 충분하면 `False`로 바꿔 전체 데이터로 학습해도 됩니다.
>
> 📊 결과 수치가 들어가는 답변(과적합 판단, 최종 리포트 표)은 **실행 결과로 자동 생성되는 셀**과 연결해 두었습니다. 그래서 다시 실행해도 숫자가 결과와 어긋나지 않습니다.
>
> 🛠️ 템플릿 코드에서 고친 곳은 `[수정]`, 새로 넣은 줄은 `[추가]`로 표시했습니다.

## 📚 Part 0: 환경 설정 및 데이터 다운로드

In [1]:
# 필요한 라이브러리 import
import torch                                          # 파이토치 본체
import torch.nn as nn                                 # 층·손실 함수
import torch.optim as optim                           # 옵티마이저·학습률 스케줄러
from torch.utils.data import DataLoader, Dataset, random_split  # 배치 공급기, 데이터셋 틀, 무작위 분할
import torchvision                                    # 이미지용 파이토치 확장 (데이터셋·모델·변환)
from torchvision import transforms, models            # 이미지 변환 도구, 사전학습 모델 모음
from torchvision.datasets import ImageFolder          # '폴더 이름 = 클래스' 구조의 이미지 데이터셋

import numpy as np                                    # 수치 계산
import pandas as pd                                   # 표 데이터 (성능 비교표)
import matplotlib.pyplot as plt                       # 그래프
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report  # 평가 지표
import seaborn as sns                                 # 히트맵(혼동 행렬)
from PIL import Image                                 # 이미지 파일 다루기
import os                                             # 파일·폴더 경로
from tqdm.notebook import tqdm                        # 진행 막대 (선택 사항)
import time                                           # [추가] 모델별 학습 시간 측정
import copy                                           # [추가] 최적 가중치를 '깊은 복사'로 저장

# 재현성을 위한 시드 설정
torch.manual_seed(42)                                 # 파이토치 난수 고정
np.random.seed(42)                                    # numpy 난수 고정

# GPU 사용 가능 여부 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # GPU가 있으면 cuda
print(f'Using device: {device}')                      # 사용 장치 (Colab T4 런타임이면 cuda)

if device.type == 'cuda':                             # GPU를 쓰는 경우에만
    print(f'GPU: {torch.cuda.get_device_name(0)}')    # GPU 이름 (예: Tesla T4)
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')  # GPU 메모리 용량

Using device: cpu


In [2]:
# ============================================
# 데이터셋 선택
# ============================================
# 아래 옵션 중 하나를 선택하세요:
# - "medmnist": 의료 이미지 데이터셋 (권장 - 헬스케어 도메인!)
# - "cifar10": 일반 이미지 데이터셋 (빠른 테스트용)
# - "chest_xray": Chest X-Ray 폐렴 데이터셋 (Kaggle 다운로드 필요)

DATASET_CHOICE = "medmnist"  # 헬스케어 도메인인 MedMNIST 선택

# MedMNIST 사용 시 서브 데이터셋 선택
# - "pathmnist": 대장 조직 병리 이미지 (9 클래스)
# - "dermamnist": 피부 병변 이미지 (7 클래스)
# - "bloodmnist": 혈액 세포 이미지 (8 클래스)
# - "breastmnist": 유방 초음파 이미지 (2 클래스 - 이진 분류)
MEDMNIST_TYPE = "pathmnist"  # 대장 조직 병리 이미지 9개 클래스 (5일차 과제와 같은 데이터)

print(f"선택된 데이터셋: {DATASET_CHOICE}")  # 선택 확인

선택된 데이터셋: medmnist


### 🏥 MedMNIST: 의료 이미지 벤치마크 데이터셋

MedMNIST는 10가지 의료 이미지 데이터셋을 표준화된 28x28 형식으로 제공합니다.
- **PathMNIST**: 대장 조직 병리 이미지 (9개 클래스)
- **DermaMNIST**: 피부 병변 이미지 (7개 클래스)
- **BloodMNIST**: 혈액 세포 이미지 (8개 클래스)
- **BreastMNIST**: 유방 초음파 이미지 (2개 클래스)

**장점**:
- 헬스케어 도메인에 적합
- MNIST와 유사한 크기로 빠른 학습 가능
- 다양한 의료 이미지 유형 경험

In [3]:
# MedMNIST 설치 및 설정
if DATASET_CHOICE == "medmnist":                         # MedMNIST를 고른 경우
    # MedMNIST 라이브러리 설치
    !pip install medmnist -q

    import medmnist                                      # MedMNIST 패키지
    from medmnist import INFO                            # 데이터셋별 설명·클래스 정보 사전

    # 선택한 데이터셋 정보 출력
    info = INFO[MEDMNIST_TYPE]                           # pathmnist 정보 꺼내기
    print(f"\n📊 {MEDMNIST_TYPE.upper()} 데이터셋 정보:")
    print(f"  - 설명: {info['description']}")
    print(f"  - 클래스 수: {info['n_channels']} 채널, {len(info['label'])} 클래스")  # 채널 수와 클래스 수
    print(f"  - 클래스 목록: {info['label']}")
    print(f"  - 샘플 수: {info['n_samples']}")           # [추가] train/val/test 이미지 수

    # 데이터셋 클래스 및 클래스 이름 설정
    DataClass = getattr(medmnist, info['python_class'])  # 문자열 'PathMNIST' → 실제 데이터셋 클래스
    NUM_CLASSES = len(info['label'])                     # 클래스 수
    CLASS_NAMES = list(info['label'].values())           # 클래스 이름 리스트
    N_CHANNELS = info['n_channels']                      # 채널 수 (3 = 컬러)

    print(f"\n✅ MedMNIST 설정 완료!")
    print(f"   클래스 수: {NUM_CLASSES}")
    print(f"   채널 수: {N_CHANNELS}")

elif DATASET_CHOICE == "cifar10":                        # CIFAR-10을 고른 경우
    print("CIFAR-10 데이터셋을 사용합니다.")
    NUM_CLASSES = 10                                     # 클래스 10개
    CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
    N_CHANNELS = 3                                       # 컬러

elif DATASET_CHOICE == "chest_xray":                     # Chest X-Ray를 고른 경우
    print("Chest X-Ray 데이터셋을 사용합니다.")
    print("⚠️ Kaggle에서 데이터셋을 다운로드해야 합니다.")
    DATA_DIR = "chest_xray"  # 다운로드 후 경로 설정
    NUM_CLASSES = 2                                      # 정상 / 폐렴
    CLASS_NAMES = ['NORMAL', 'PNEUMONIA']
    N_CHANNELS = 3


📊 PATHMNIST 데이터셋 정보:
  - 설명: The PathMNIST is based on a prior study for predicting survival from colorectal cancer histology slides, providing a dataset (NCT-CRC-HE-100K) of 100,000 non-overlapping image patches from hematoxylin & eosin stained histological images, and a test dataset (CRC-VAL-HE-7K) of 7,180 image patches from a different clinical center. The dataset is comprised of 9 types of tissues, resulting in a multi-class classification task. We resize the source images of 3×224×224 into 3×28×28, and split NCT-CRC-HE-100K into training and validation set with a ratio of 9:1. The CRC-VAL-HE-7K is treated as the test set.
  - 클래스 수: 3 채널, 9 클래스
  - 클래스 목록: {'0': 'adipose', '1': 'background', '2': 'debris', '3': 'lymphocytes', '4': 'mucus', '5': 'smooth muscle', '6': 'normal colon mucosa', '7': 'cancer-associated stroma', '8': 'colorectal adenocarcinoma epithelium'}
  - 샘플 수: {'train': 89996, 'val': 10004, 'test': 7180}

✅ MedMNIST 설정 완료!
   클래스 수: 9
   채널 수: 3


## 📊 Part 1: 데이터 로드 및 전처리

In [4]:
# ============================================
# 이미지 변환(Transform) 정의
# ============================================
# ImageNet 정규화 값 (사전학습 모델과 동일하게 맞춤)
IMAGENET_MEAN = [0.485, 0.456, 0.406]   # ImageNet 사진들의 R, G, B 평균 → 사전학습 모델이 보던 입력 분포와 맞춤
IMAGENET_STD = [0.229, 0.224, 0.225]    # R, G, B 표준편차

# Chest X-Ray용 변환 (학습 - 데이터 증강 포함)
augment_transform = transforms.Compose([                  # 여러 변환을 순서대로 적용하는 묶음
    transforms.Resize((256, 256)),                        # 256x256으로 크기 조정
    transforms.RandomCrop(224),                           # 무작위 위치에서 224x224 잘라내기 (증강)
    transforms.RandomHorizontalFlip(p=0.5),               # 50% 확률로 좌우 반전 (증강)
    transforms.RandomRotation(15),                        # -15~15도 무작위 회전 (증강)
    transforms.ColorJitter(brightness=0.2, contrast=0.2), # 밝기·대비 무작위 변화 (증강)
    transforms.ToTensor(),                                # PIL 이미지 → 텐서 (0~1 값, [C, H, W])
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)  # 채널별 (x - 평균) / 표준편차
])

# Chest X-Ray용 변환 (테스트 - 증강 없음)
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),                        # 크기 조정만
    transforms.ToTensor(),                                # 텐서 변환
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)  # 정규화
])

In [ ]:
# ============================================
# 데이터셋 로드 (선택에 따라 다른 데이터셋 사용)
# ============================================

def to_rgb(x):
    """그레이스케일(1채널)을 RGB(3채널)로 변환"""
    if x.shape[0] == 1:                    # 채널이 1개(흑백)이면
        return x.repeat(3, 1, 1)           # 같은 값을 3번 복사해 3채널로 (사전학습 모델은 3채널 입력을 기대)
    return x                               # 이미 3채널이면 그대로


if DATASET_CHOICE == "medmnist":
    # -----------------------------------------
    # MedMNIST 데이터셋 (의료 이미지)
    # 28x28 → 224x224 리사이즈 (전이학습 모델 호환)
    # -----------------------------------------
    medmnist_transform_train = transforms.Compose([
        transforms.ToTensor(),                     # PIL → 텐서 [C, 28, 28]
        transforms.Lambda(to_rgb),                 # 1채널 → 3채널
        transforms.Resize((224, 224)),             # 28 → 224로 확대 (ResNet 입력 크기에 맞춤)
        transforms.RandomHorizontalFlip(p=0.5),    # 좌우 반전 (증강)
        transforms.RandomRotation(15),             # ±15도 회전 (증강)
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)  # ImageNet 기준 정규화
    ])

    medmnist_transform_test = transforms.Compose([
        transforms.ToTensor(),                     # 텐서 변환
        transforms.Lambda(to_rgb),                 # 3채널로
        transforms.Resize((224, 224)),             # 크기만 맞춤 (증강 없음)
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    train_dataset = DataClass(split='train', transform=medmnist_transform_train, download=True)  # 학습용 (처음 실행 시 다운로드)
    val_dataset = DataClass(split='val', transform=medmnist_transform_test, download=True)       # 검증용
    test_dataset = DataClass(split='test', transform=medmnist_transform_test, download=True)     # 테스트용

    print(f"\n✅ {MEDMNIST_TYPE.upper()} 데이터셋 로드 완료!")

elif DATASET_CHOICE == "cifar10":
    # -----------------------------------------
    # CIFAR-10 데이터셋 (일반 이미지)
    # -----------------------------------------
    cifar_transform_train = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    cifar_transform_test = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    full_train_dataset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=cifar_transform_train
    )
    test_dataset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=cifar_transform_test
    )

    # Train/Val 분할 (80:20)
    train_size = int(0.8 * len(full_train_dataset))        # 학습 80%
    val_size = len(full_train_dataset) - train_size        # 나머지 20%는 검증
    train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

    print("\n✅ CIFAR-10 데이터셋 로드 완료!")

elif DATASET_CHOICE == "chest_xray":
    # -----------------------------------------
    # Chest X-Ray 데이터셋 (Kaggle 다운로드 필요)
    # -----------------------------------------
    train_dataset = ImageFolder(f"{DATA_DIR}/train", transform=augment_transform)
    val_dataset = ImageFolder(f"{DATA_DIR}/val", transform=test_transform)
    test_dataset = ImageFolder(f"{DATA_DIR}/test", transform=test_transform)

    print("\n✅ Chest X-Ray 데이터셋 로드 완료!")

# 데이터셋 크기 출력
print(f"Train set: {len(train_dataset):,} samples")
print(f"Validation set: {len(val_dataset):,} samples")
print(f"Test set: {len(test_dataset):,} samples")

#### ⏱️ [추가] 빠른 실습 모드 (QUICK_MODE)

28×28 이미지를 224×224로 키우면 이미지 한 장의 계산량이 (224/28)² = 64배가 됩니다. 학습 이미지 89,996장 전체로 4개 모델을 학습하면 GPU에서도 오래 걸리고 무료 Colab은 장시간 실행 중 연결이 끊길 수 있어서, 기본값으로는 **학습 20,000장(전체의 약 22%) / 검증 4,000장**만 무작위로(시드 고정) 뽑아 씁니다. **테스트 세트는 전체**를 그대로 쓰므로 모델끼리의 비교 기준은 같습니다.

In [ ]:
from torch.utils.data import Subset           # 데이터셋에서 일부 인덱스만 골라 쓰는 도구

QUICK_MODE = True                             # 빠른 실습 모드 켜기(True) / 끄기(False)
N_TRAIN, N_VAL = 20000, 4000                  # 사용할 학습·검증 이미지 수 (32의 배수 → 크기 1짜리 마지막 배치가 생기지 않음)

if QUICK_MODE:
    g = torch.Generator().manual_seed(42)     # 뽑기용 난수 생성기 (항상 같은 이미지가 뽑히도록 고정)
    train_idx = torch.randperm(len(train_dataset), generator=g)[:min(N_TRAIN, len(train_dataset))]  # 학습 인덱스를 섞어 앞에서 N개
    val_idx = torch.randperm(len(val_dataset), generator=g)[:min(N_VAL, len(val_dataset))]          # 검증 인덱스 N개
    train_dataset = Subset(train_dataset, train_idx.tolist())   # 학습 데이터셋을 부분집합으로 교체
    val_dataset = Subset(val_dataset, val_idx.tolist())         # 검증 데이터셋을 부분집합으로 교체

print(f"QUICK_MODE={QUICK_MODE} → Train {len(train_dataset):,} / Val {len(val_dataset):,} / Test {len(test_dataset):,}")

In [ ]:
# DataLoader 생성
BATCH_SIZE = 32                                       # 한 번에 32장씩 학습

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)   # 학습: 에폭마다 섞어서 공급
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)      # 검증: 순서대로
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)    # 테스트: 순서대로
# num_workers=0: 데이터 준비를 메인 프로세스 하나가 담당 (가장 단순하고 안전한 설정)

print(f"Number of batches - Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")  # 에폭당 배치 수

In [ ]:
# 샘플 이미지 시각화
def imshow(img, title=None):
    """이미지 시각화 (정규화 역변환 포함)"""
    img = img.numpy().transpose((1, 2, 0))           # 텐서 [C, H, W] → numpy [H, W, C] (matplotlib 형식)
    mean = np.array([0.485, 0.456, 0.406])           # 정규화에 썼던 평균
    std = np.array([0.229, 0.224, 0.225])            # 정규화에 썼던 표준편차
    img = std * img + mean                           # 정규화 되돌리기: x * std + mean
    img = np.clip(img, 0, 1)                         # 0~1 범위를 벗어난 값 자르기
    plt.imshow(img)                                  # '현재 그림 칸'에 이미지 표시
    if title:
        plt.title(title)

# 샘플 배치 가져오기
images, labels = next(iter(train_loader))            # 학습 로더에서 첫 배치(32장) 꺼내기

fig, axes = plt.subplots(2, 4, figsize=(12, 6))      # 2행 4열 = 8칸
for i, ax in enumerate(axes.flat):                   # 8칸을 하나씩
    ax.axis('off')                                   # 축 눈금 숨기기
    plt.sca(ax)                                      # [수정] 현재 칸을 ax로 지정 → 없으면 plt.imshow가 8장 모두 마지막 칸에 겹쳐 그림
    imshow(images[i])                                # i번째 이미지 표시
    ax.set_title(f'{CLASS_NAMES[labels[i]]}')        # 라벨 번호로 클래스 이름 찾아 제목 달기

plt.suptitle('Sample Training Images', fontsize=14)  # 전체 제목
plt.tight_layout()
plt.show()

---

## 🧠 Part 2: 기본 CNN 모델 구현 (Day 4 복습)

In [5]:
class SimpleCNN(nn.Module):
    """기본 CNN 모델"""
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()                        # 부모 초기화

        # 합성곱 레이어
        self.conv_layers = nn.Sequential(
            # Block 1: 224 -> 112
            nn.Conv2d(3, 32, kernel_size=3, padding=1),          # 3채널 → 32개 특징맵, 3x3 필터, 테두리 1칸 채움(크기 유지)
            nn.BatchNorm2d(32),                                  # 채널별 정규화 → 학습 안정화
            nn.ReLU(),                                           # 비선형 활성화
            nn.MaxPool2d(2, 2),                                  # 2x2 칸에서 최댓값만 → 가로·세로 절반

            # Block 2: 112 -> 56
            nn.Conv2d(32, 64, kernel_size=3, padding=1),         # 32 → 64채널
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 3: 56 -> 28
            nn.Conv2d(64, 128, kernel_size=3, padding=1),        # 64 → 128채널
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            # Block 4: 28 -> 14
            nn.Conv2d(128, 256, kernel_size=3, padding=1),       # 128 → 256채널
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        # Global Average Pooling
        self.gap = nn.AdaptiveAvgPool2d((1, 1))                  # 각 채널(14x14)을 평균 1개 값으로 요약 → [256, 1, 1]

        # 분류기
        self.classifier = nn.Sequential(
            nn.Flatten(),                                        # [256, 1, 1] → [256]
            nn.Dropout(0.5),                                     # 학습 때 50% 뉴런 끄기 (과적합 방지)
            nn.Linear(256, 128),                                 # 256 → 128
            nn.ReLU(),
            nn.Dropout(0.3),                                     # 30% 끄기
            nn.Linear(128, num_classes)                          # 128 → 클래스 수 (각 클래스의 점수 = 로짓)
        )

    def forward(self, x):
        x = self.conv_layers(x)                                  # 특징 추출
        x = self.gap(x)                                          # 전역 평균 풀링
        x = self.classifier(x)                                   # 분류
        return x                                                 # 로짓 반환 (Softmax는 CrossEntropyLoss 안에서 처리)

#### 🔍 [추가] 특징맵 크기 직접 추적하기

데이터나 GPU 없이도, **가짜 이미지 1장**(0으로 채운 3×224×224 텐서)을 모델에 통과시키면 블록마다 특징맵 모양이 어떻게 바뀌는지 확인할 수 있습니다. 과제 2-1의 1번·2번 답의 근거가 되는 셀입니다.

In [6]:
probe = SimpleCNN(num_classes=NUM_CLASSES)           # 구조 확인용 모델 하나 생성
probe.eval()                                         # 평가 모드 (BatchNorm이 이미지 1장으로도 동작하도록)
x = torch.zeros(1, 3, 224, 224)                      # 가짜 이미지 1장: [배치, 채널, 높이, 너비]
print(f"입력          : {tuple(x.shape)}")
with torch.no_grad():                                # 기울기 계산 없이
    block = 0                                        # 블록 번호
    for layer in probe.conv_layers:                  # 합성곱 부분의 층을 하나씩 통과시키며
        x = layer(x)                                 # 한 층 통과
        if isinstance(layer, nn.MaxPool2d):          # 블록의 마지막(풀링)을 지날 때마다 모양 출력
            block += 1
            print(f"Block {block} 출력  : {tuple(x.shape)}  → 채널 {x.shape[1]}, 크기 {x.shape[2]}x{x.shape[3]}")
    x = probe.gap(x)                                 # Global Average Pooling
    print(f"GAP 출력      : {tuple(x.shape)}")
    x = probe.classifier(x)                          # 분류기
    print(f"분류기 출력   : {tuple(x.shape)}  → 클래스 {NUM_CLASSES}개의 점수(로짓)")

# GAP 대신 Flatten으로 바로 펼쳤다면 첫 Linear 층의 파라미터는 몇 개일까?
flat = 256 * 14 * 14                                 # 마지막 특징맵을 그대로 펼친 길이
print(f"\nFlatten 길이 = 256 x 14 x 14 = {flat:,}")
print(f"Flatten 후 Linear({flat:,}, 128) 파라미터: {flat * 128 + 128:,}개")
print(f"GAP 후     Linear(256, 128) 파라미터: {256 * 128 + 128:,}개")
print(f"SimpleCNN 전체 파라미터: {sum(p.numel() for p in probe.parameters()):,}개")

입력          : (1, 3, 224, 224)


Block 1 출력  : (1, 32, 112, 112)  → 채널 32, 크기 112x112


Block 2 출력  : (1, 64, 56, 56)  → 채널 64, 크기 56x56
Block 3 출력  : (1, 128, 28, 28)  → 채널 128, 크기 28x28


Block 4 출력  : (1, 256, 14, 14)  → 채널 256, 크기 14x14


GAP 출력      : (1, 256, 1, 1)
분류기 출력   : (1, 9)  → 클래스 9개의 점수(로짓)

Flatten 길이 = 256 x 14 x 14 = 50,176
Flatten 후 Linear(50,176, 128) 파라미터: 6,422,656개
GAP 후     Linear(256, 128) 파라미터: 32,896개
SimpleCNN 전체 파라미터: 423,433개


In [ ]:
# ============================================
# 학습 및 검증 함수
# ============================================

def train_epoch(model, train_loader, criterion, optimizer, device):
    """
    한 에폭 학습 수행

    Args:
        model: 학습할 모델
        train_loader: 학습 데이터 로더
        criterion: 손실 함수
        optimizer: 옵티마이저
        device: 연산 장치 (cuda/cpu)

    Returns:
        (평균 손실, 정확도)
    """
    model.train()                                   # 학습 모드 (Dropout·BatchNorm 학습용 동작)
    running_loss = 0.0                              # 손실 합계
    correct = 0                                     # 맞힌 개수
    total = 0                                       # 본 이미지 수

    for images, labels in train_loader:             # 배치 반복
        images = images.to(device)                  # 이미지를 GPU/CPU로
        # MedMNIST는 레이블이 (N, 1) 형태이므로 squeeze 처리
        if len(labels.shape) > 1:                   # [N, 1] 모양이면
            labels = labels.squeeze().long()        # [N]으로 펴고 정수형으로 (CrossEntropyLoss 요구 형식)
        labels = labels.to(device)                  # 라벨도 같은 장치로

        # Forward → Loss → Backward → Update
        optimizer.zero_grad()                       # 기울기 초기화
        outputs = model(images)                     # 순전파: [N, 클래스 수] 점수
        loss = criterion(outputs, labels)           # 손실 계산
        loss.backward()                             # 역전파
        optimizer.step()                            # 가중치 업데이트

        # 통계 누적
        running_loss += loss.item()                 # 손실 누적
        _, predicted = outputs.max(1)               # 점수가 가장 큰 클래스 = 예측
        total += labels.size(0)                     # 이미지 수 누적
        correct += predicted.eq(labels).sum().item()  # 맞힌 개수 누적

    return running_loss / len(train_loader), correct / total   # (배치 평균 손실, 정확도)


def validate(model, val_loader, criterion, device):
    """
    검증 수행 (gradient 계산 없음)

    Args:
        model: 평가할 모델
        val_loader: 검증 데이터 로더
        criterion: 손실 함수
        device: 연산 장치 (cuda/cpu)

    Returns:
        (평균 손실, 정확도)
    """
    model.eval()                                    # 평가 모드 (Dropout 끄기, BatchNorm 고정)
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():  # gradient 계산 비활성화 → 메모리·시간 절약
        for images, labels in val_loader:           # 검증 배치 반복
            images = images.to(device)
            if len(labels.shape) > 1:               # [N, 1] → [N]
                labels = labels.squeeze().long()
            labels = labels.to(device)

            outputs = model(images)                 # 예측
            loss = criterion(outputs, labels)       # 손실

            running_loss += loss.item()
            _, predicted = outputs.max(1)           # 예측 클래스
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    return running_loss / len(val_loader), correct / total

In [ ]:
# ============================================
# 전체 학습 루프
# ============================================

def train_model(model, train_loader, val_loader, criterion, optimizer,
                scheduler=None, num_epochs=20, patience=5, device='cpu'):
    """
    전체 학습 루프 (Early Stopping 포함)

    Args:
        model: 학습할 모델
        train_loader: 학습 데이터 로더
        val_loader: 검증 데이터 로더
        criterion: 손실 함수
        optimizer: 옵티마이저
        scheduler: 학습률 스케줄러 (선택)
        num_epochs: 최대 에폭 수
        patience: Early Stopping 인내 횟수
        device: 연산 장치 (cuda/cpu)

    Returns:
        (학습된 모델, 학습 히스토리)
    """
    model = model.to(device)                               # 모델을 GPU/CPU로 이동

    # 학습 히스토리 저장
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }

    # Early Stopping 변수
    best_val_acc = 0.0                                     # 최고 검증 정확도
    patience_counter = 0                                   # 개선 없는 연속 에폭 수
    best_model_state = None                                # 최고 시점 가중치

    for epoch in range(num_epochs):
        # 학습 & 검증
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)  # 1에폭 학습
        val_loss, val_acc = validate(model, val_loader, criterion, device)                      # 검증

        # 히스토리 기록
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        # 스케줄러 업데이트
        if scheduler:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)                   # 검증 손실이 정체되면 학습률을 낮추는 방식
            else:
                scheduler.step()                           # 정해진 에폭마다 학습률을 낮추는 방식 (StepLR 등)

        # Early Stopping 체크
        if val_acc > best_val_acc:                         # 검증 정확도 최고 기록 경신
            best_val_acc = val_acc
            best_model_state = copy.deepcopy(model.state_dict())  # [수정] .copy()는 얕은 복사라 이후 학습으로 값이 같이 바뀜 → 깊은 복사
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:               # patience번 연속 개선 없음 → 중단
                print(f"⏹️ Early stopping at epoch {epoch+1}")
                break

        # 진행 상황 출력 (5 에폭마다)
        if (epoch + 1) % 5 == 0:
            current_lr = optimizer.param_groups[0]['lr']   # 현재 학습률
            print(f"Epoch [{epoch+1:2d}/{num_epochs}] | "
                  f"Train: {train_loss:.4f}/{train_acc:.4f} | "
                  f"Val: {val_loss:.4f}/{val_acc:.4f} | "
                  f"LR: {current_lr:.6f}")

    # 최적 모델 복원
    if best_model_state:
        model.load_state_dict(best_model_state)            # 최고 검증 정확도 시점으로 되돌리기
        print(f"✅ Best model restored (Val Acc: {best_val_acc:.4f})")

    return model, history

In [ ]:
# 기본 CNN 모델 학습
print("=" * 50)
print("기본 CNN 모델 학습")
print("=" * 50)

train_times = {}                                     # [추가] 모델별 학습 시간(초) → 최종 리포트 표에 사용
torch.manual_seed(42)                                # [추가] 재현성: 초기 가중치·배치 순서 고정

simple_cnn = SimpleCNN(num_classes=NUM_CLASSES)      # 모델 생성
criterion = nn.CrossEntropyLoss()                    # 다중 분류 손실 (내부에서 Softmax + 로그 손실 계산)
optimizer = optim.Adam(simple_cnn.parameters(), lr=0.001)   # Adam, 학습률 0.001
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)  # 검증 손실이 3에폭 정체되면 학습률 절반

start_time = time.time()                             # [추가] 학습 시작 시각
simple_cnn, simple_history = train_model(
    simple_cnn, train_loader, val_loader,
    criterion, optimizer, scheduler,
    num_epochs=20, patience=7, device=device         # 최대 20에폭, 7에폭 연속 개선 없으면 중단
)
train_times['Simple CNN'] = time.time() - start_time # [추가] 걸린 시간 저장
print(f"⏱️ 학습 시간: {train_times['Simple CNN'] / 60:.1f}분")

In [ ]:
# 학습 곡선 시각화
def plot_history(history, title="Training History"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))                     # 왼쪽 손실, 오른쪽 정확도

    axes[0].plot(history['train_loss'], label='Train', color='#8FBC8F')  # 학습 손실
    axes[0].plot(history['val_loss'], label='Val', color='#87CEEB')      # 검증 손실
    axes[0].set_title(f'{title} - Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history['train_acc'], label='Train', color='#8FBC8F')   # 학습 정확도
    axes[1].plot(history['val_acc'], label='Val', color='#87CEEB')       # 검증 정확도
    axes[1].set_title(f'{title} - Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_history(simple_history, "Simple CNN")           # Simple CNN 학습 곡선

#### 🔍 [추가] 학습 곡선 자동 진단

과제 2-1의 3번(과적합 판단)을 눈대중이 아니라 **실제 학습 기록의 숫자**로 판단하는 셀입니다. 기준(경험칙): ① 마지막 에폭의 `학습 정확도 − 검증 정확도`가 0.05보다 큰지, ② 검증 손실이 최저점 이후 0.05보다 많이 다시 올랐는지.

In [ ]:
def diagnose(history, name):
    """학습 기록으로 과적합 여부를 진단해 결론 문장을 출력"""
    best_e = int(np.argmax(history['val_acc']))                           # 검증 정확도 최고 에폭(0부터)
    min_e = int(np.argmin(history['val_loss']))                           # 검증 손실 최저 에폭(0부터)
    gap = history['train_acc'][-1] - history['val_acc'][-1]               # 마지막 에폭 (학습 - 검증) 정확도 차이
    rebound = history['val_loss'][-1] - history['val_loss'][min_e]        # 검증 손실 최저점 대비 반등폭
    print(f"[{name}] 총 {len(history['val_acc'])}에폭 학습")
    print(f"  최고 Val Acc {history['val_acc'][best_e]:.4f} ({best_e + 1}에폭) | 최저 Val Loss {history['val_loss'][min_e]:.4f} ({min_e + 1}에폭)")
    print(f"  마지막 에폭: Train Acc {history['train_acc'][-1]:.4f} / Val Acc {history['val_acc'][-1]:.4f} → 차이 {gap:+.4f}")
    print(f"  검증 손실 반등폭: {rebound:+.4f}")
    if gap > 0.05 and rebound > 0.05:                                     # 두 신호 모두
        verdict = "과적합 징후가 뚜렷합니다 (학습 정확도만 높고, 검증 손실이 최저점 이후 다시 올랐습니다)."
    elif gap > 0.05 or rebound > 0.05:                                    # 신호 하나만
        verdict = "약한 과적합 징후가 있습니다 (학습-검증 차이 또는 검증 손실 반등 중 하나가 나타났습니다)."
    elif gap < -0.02:                                                     # 검증이 오히려 더 높음
        verdict = ("과적합은 아닙니다. 오히려 검증 정확도가 더 높은데, Dropout·데이터 증강이 학습 때만 켜져 "
                   "학습 점수가 낮게 측정되기 때문입니다.")
    else:
        verdict = "과적합 징후가 거의 없습니다 (학습·검증 곡선이 함께 움직입니다)."
    print("  → 진단:", verdict)

diagnose(simple_history, "Simple CNN")               # Simple CNN 진단

### ✏️ [과제 2-1] CNN 구조 분석

아래 질문에 답하세요:

1. SimpleCNN 모델에서 각 Conv 블록을 거칠 때마다 특징맵의 크기가 어떻게 변하나요?
   - 답변: 각 블록의 `Conv2d(kernel_size=3, padding=1)`는 출력 크기 공식 `(W − K + 2P) / S + 1 = (224 − 3 + 2) / 1 + 1 = 224`처럼 **가로·세로를 그대로 유지**하고 채널 수만 바꾸며, 이어지는 `MaxPool2d(2, 2)`가 **가로·세로를 절반**으로 줄입니다. 그래서 위 '특징맵 크기 추적' 셀의 출력처럼 [채널, 높이, 너비]가 `[3, 224, 224]` → Block 1 `[32, 112, 112]` → Block 2 `[64, 56, 56]` → Block 3 `[128, 28, 28]` → Block 4 `[256, 14, 14]`로 변하고, GAP를 지나면 `[256, 1, 1]`, 분류기를 지나면 클래스 수(9)만큼의 점수가 됩니다. 블록마다 **면적은 1/4로 줄고 채널은 2배로** 늘어납니다.

2. Global Average Pooling(GAP)을 사용하는 이유는 무엇인가요?
   - 답변: GAP는 각 채널(14×14 특징맵)의 **평균값 하나만** 남겨 256×14×14를 256개 숫자로 요약합니다.
     - ① **파라미터 절약·과적합 방지**: Flatten으로 펼치면 50,176개 입력을 받는 `Linear(50176, 128)`에 6,422,656개 파라미터가 필요하지만, GAP 뒤에는 `Linear(256, 128)`의 32,896개면 충분합니다(GAP 자체는 학습할 파라미터가 0개). 이 모델 전체 파라미터(위 출력 기준)보다도 훨씬 큰 층을 없앤 셈입니다.
     - ② **입력 크기 유연성**: `AdaptiveAvgPool2d((1, 1))`는 특징맵 크기가 달라도 항상 1×1로 만들어 주므로 다른 해상도의 이미지도 같은 분류기로 처리할 수 있습니다.
     - ③ **위치에 덜 민감**: "이 특징이 이미지 어딘가에 얼마나 있나"를 요약하므로 조직 패턴의 위치가 조금 달라도 결과가 안정적입니다.

3. 학습 곡선을 보고 과적합 여부를 판단하세요.
   - 답변: 판단 기준은 ① 학습 정확도는 계속 오르는데 검증 정확도가 정체·하락하는지, ② 검증 손실이 최저점을 찍은 뒤 다시 오르는지입니다. 위 **'학습 곡선 자동 진단'** 셀이 Simple CNN의 실제 학습 기록으로 두 가지를 계산해 `→ 진단:` 문장으로 결론을 출력하므로, **그 문장이 이 질문의 답**입니다(다시 실행해도 그래프·결론이 항상 일치).
     - 해석할 때 참고: 이 모델에는 Dropout(0.5, 0.3)·BatchNorm·데이터 증강·Early Stopping(최고 검증 정확도 시점 복원)처럼 과적합을 억제하는 장치가 여러 개 들어 있습니다. 또 Dropout과 증강은 **학습 때만** 켜지기 때문에, 학습 정확도가 검증 정확도보다 낮게 나오는 것은 과적합이 아니라 정상적인 현상입니다.

---

## 🔄 Part 3: 전이학습 - Feature Extraction (Day 5 예습)

사전학습된 모델을 특징 추출기로 사용합니다.

In [ ]:
# ============================================
# Feature Extraction 모델 생성
# ============================================

def create_feature_extractor(num_classes=10):
    """
    Feature Extraction 방식의 전이학습 모델 생성
    - 사전학습된 가중치를 동결하고 마지막 분류기만 학습
    """
    # 1. 사전학습된 ResNet18 로드
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)  # ImageNet 120만 장으로 학습된 가중치 (처음엔 다운로드)

    # 2. 모든 레이어 동결 (가중치 업데이트 X)
    for param in model.parameters():
        param.requires_grad = False                  # 기울기를 계산하지 않음 → 옵티마이저가 바꾸지 못함

    # 3. 마지막 FC 레이어만 교체 (이 부분만 학습됨)
    num_features = model.fc.in_features  # 512       # 원래 fc의 입력 크기
    model.fc = nn.Sequential(                        # 새로 만든 층은 기본값이 requires_grad=True → 학습됨
        nn.Dropout(0.5),                             # 과적합 방지
        nn.Linear(num_features, num_classes)         # 512 → 클래스 수 (ImageNet의 1000개 대신)
    )

    return model

# 모델 생성 및 파라미터 수 확인
feature_extractor = create_feature_extractor(NUM_CLASSES)
trainable_params = sum(p.numel() for p in feature_extractor.parameters() if p.requires_grad)  # 학습 가능한 파라미터
total_params = sum(p.numel() for p in feature_extractor.parameters())                         # 전체 파라미터
print(f"Feature Extractor - Trainable: {trainable_params:,} / Total: {total_params:,} "
      f"({trainable_params/total_params*100:.2f}%)")

In [ ]:
# Feature Extraction 학습
print("=" * 50)
print("Feature Extraction 학습")
print("=" * 50)

torch.manual_seed(42)                                                  # [추가] 재현성
feature_extractor = create_feature_extractor(NUM_CLASSES)              # 새 모델
criterion = nn.CrossEntropyLoss()                                      # 손실 함수
optimizer = optim.Adam(feature_extractor.fc.parameters(), lr=0.001)    # fc(새 분류기)만 업데이트
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)  # 5에폭마다 학습률 절반

start_time = time.time()                                               # [추가] 시간 측정 시작
feature_extractor, fe_history = train_model(
    feature_extractor, train_loader, val_loader,
    criterion, optimizer, scheduler,
    num_epochs=15, patience=5, device=device
)
train_times['Feature Extraction'] = time.time() - start_time           # [추가] 학습 시간 저장

plot_history(fe_history, "Feature Extraction")                         # 학습 곡선

---

## 🎯 Part 4: 전이학습 - Fine-tuning

사전학습된 모델의 일부 레이어도 함께 학습합니다.

In [ ]:
# ============================================
# Fine-tuning 모델 생성
# ============================================

def create_finetuning_model(num_classes=10, unfreeze_layers=['layer4', 'fc']):
    """
    Fine-tuning 방식의 전이학습 모델 생성
    - Feature Extraction과 달리 일부 레이어도 함께 학습
    """
    # 1. 사전학습된 ResNet18 로드
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # 2. 모든 레이어 동결
    for param in model.parameters():
        param.requires_grad = False

    # 3. 지정된 레이어만 동결 해제 (layer4, fc)
    for name, module in model.named_children():          # 큰 덩어리(conv1, layer1~4, fc 등)를 이름과 함께 순회
        if name in unfreeze_layers:                      # 풀어 줄 목록에 있으면
            for param in module.parameters():
                param.requires_grad = True               # 다시 학습 가능하게

    # 4. 마지막 FC 레이어 교체
    num_features = model.fc.in_features                  # 512
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_features, num_classes)
    )

    return model

# 모델 생성 및 파라미터 수 확인
finetuning_model = create_finetuning_model(NUM_CLASSES)
trainable_params = sum(p.numel() for p in finetuning_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in finetuning_model.parameters())
print(f"Fine-tuning - Trainable: {trainable_params:,} / Total: {total_params:,} "
      f"({trainable_params/total_params*100:.2f}%)")

#### 📐 [추가] 학습 가능한 파라미터 수를 층별로 분해해 보기

파라미터 **개수**는 가중치 값과 상관없이 **구조**로만 정해집니다. 그래서 사전학습 가중치를 내려받지 않고(`weights=None`) 같은 구조를 만들어 세어도 위 두 셀과 같은 숫자가 나옵니다. 과제 4-1의 2번 답의 근거입니다.

In [7]:
def count_params(model):
    """(학습 가능한 파라미터 수, 전체 파라미터 수)"""
    return (sum(p.numel() for p in model.parameters() if p.requires_grad),
            sum(p.numel() for p in model.parameters()))

def resnet18_with_head(num_classes, unfreeze=()):
    """위 두 함수와 같은 구조를 가중치 다운로드 없이 생성"""
    m = models.resnet18(weights=None)                    # 구조만 생성 (값은 무작위지만 개수는 동일)
    for p in m.parameters():
        p.requires_grad = False                          # 전부 동결
    for name, module in m.named_children():
        if name in unfreeze:                             # 지정한 블록만 동결 해제
            for p in module.parameters():
                p.requires_grad = True
    m.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(m.fc.in_features, num_classes))  # 새 분류기
    return m

fe_t, fe_all = count_params(resnet18_with_head(NUM_CLASSES))                         # Feature Extraction 구조
ft_t, ft_all = count_params(resnet18_with_head(NUM_CLASSES, ('layer4', 'fc')))       # Fine-tuning 구조
layer4_n = sum(p.numel() for p in models.resnet18(weights=None).layer4.parameters()) # layer4 블록의 파라미터 수

print(f"새 fc 층       : 512 x {NUM_CLASSES} + {NUM_CLASSES} = {512 * NUM_CLASSES + NUM_CLASSES:,}개")
print(f"layer4 블록    : {layer4_n:,}개")
print(f"Feature Extraction 학습 가능: {fe_t:>10,} / 전체 {fe_all:,} ({fe_t / fe_all * 100:.2f}%)")
print(f"Fine-tuning        학습 가능: {ft_t:>10,} / 전체 {ft_all:,} ({ft_t / ft_all * 100:.2f}%)")
print(f"→ Fine-tuning이 학습하는 파라미터가 약 {ft_t / fe_t:,.0f}배 많습니다.")

새 fc 층       : 512 x 9 + 9 = 4,617개
layer4 블록    : 8,393,728개
Feature Extraction 학습 가능:      4,617 / 전체 11,181,129 (0.04%)
Fine-tuning        학습 가능:  8,398,345 / 전체 11,181,129 (75.11%)
→ Fine-tuning이 학습하는 파라미터가 약 1,819배 많습니다.


In [ ]:
# Fine-tuning 학습
print("=" * 50)
print("Fine-tuning 학습")
print("=" * 50)

torch.manual_seed(42)                                         # [추가] 재현성
finetuning_model = create_finetuning_model(NUM_CLASSES)       # 새 모델
criterion = nn.CrossEntropyLoss()

# 레이어별 다른 학습률 적용
optimizer = optim.Adam([
    {'params': finetuning_model.layer4.parameters(), 'lr': 0.0001},   # 사전학습된 layer4: 작은 학습률로 살살 조정
    {'params': finetuning_model.fc.parameters(), 'lr': 0.001}         # 새로 만든 fc: 10배 큰 학습률로 빠르게 학습
])
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)  # 5에폭마다 두 학습률 모두 절반

start_time = time.time()                                      # [추가] 시간 측정 시작
finetuning_model, ft_history = train_model(
    finetuning_model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    num_epochs=15, patience=5, device=device
)
train_times['Fine-tuning'] = time.time() - start_time         # [추가] 학습 시간 저장

plot_history(ft_history, "Fine-tuning")

### ✏️ [과제 4-1] Feature Extraction vs Fine-tuning 비교

아래 질문에 답하세요:

1. Feature Extraction과 Fine-tuning의 차이점을 설명하세요.
   - 답변: **Feature Extraction**은 사전학습된 ResNet18의 합성곱 부분(이미지에서 특징을 뽑는 부분) 전체를 **동결**하고, 맨 끝에 새로 붙인 분류기(fc)만 학습합니다. **Fine-tuning**은 분류기와 함께 **사전학습된 층 일부(여기서는 마지막 블록 `layer4`)도 동결을 풀어** 새 데이터에 맞게 조금씩 고칩니다. 즉 FE는 "ImageNet에서 배운 특징을 그대로 쓰고 판단 기준만 새로 배우기", FT는 "특징 자체도 병리 이미지에 맞게 다듬기"입니다.
   - 💡 비유: FE는 숙련된 사진작가의 '보는 눈'은 그대로 빌리고 분류 기준표만 새로 만드는 것이고, FT는 그 작가에게 병리 슬라이드 보는 훈련을 추가로 시키는 것입니다.

2. 학습 가능한 파라미터 수가 다른 이유는 무엇인가요?
   - 답변: 옵티마이저는 `requires_grad=True`인 파라미터만 업데이트하는데, 두 방식은 **동결을 푼 범위가 다르기** 때문입니다. 위 '층별 분해' 셀 출력 기준으로 FE는 새 fc(512×9 + 9 = **4,617개**)만 학습해 전체 11,181,129개 중 **0.04%**이고, FT는 여기에 layer4(**8,393,728개**)가 더해져 **8,398,345개(75.11%)**를 학습합니다(약 1,819배). layer4는 채널이 512개인 3×3 합성곱들로 이뤄져 ResNet18 파라미터의 대부분이 몰려 있는 블록이라, 이 블록 하나만 풀어도 학습량이 크게 늘어납니다.

3. Fine-tuning에서 레이어별로 다른 학습률을 사용하는 이유는 무엇인가요?
   - 답변: `layer4`는 ImageNet으로 **이미 잘 학습된** 가중치라서 큰 학습률로 크게 바꾸면 그동안 배운 유용한 특징이 망가질 수 있습니다(파국적 망각, catastrophic forgetting). 그래서 **0.0001로 살살** 조정합니다. 반면 새로 붙인 `fc`는 **무작위 초기값**에서 시작하므로 **0.001(10배)로 빠르게** 배워야 합니다.
   - 💡 비유: 경력직 직원(layer4)은 회사 방식에 맞게 '조금씩' 적응시키고, 신입 직원(fc)은 처음부터 '많이' 가르치는 것과 같습니다.

4. 어떤 상황에서 Feature Extraction이 더 적합하고, 어떤 상황에서 Fine-tuning이 더 적합한가요?
   - 답변:
     - **Feature Extraction이 적합**: ① 데이터가 적을 때(학습할 파라미터가 적어 과적합 위험이 낮음) ② 새 데이터가 사전학습 데이터(ImageNet 일상 사진)와 비슷할 때 ③ GPU·시간이 부족하거나 빠른 베이스라인이 필요할 때.
     - **Fine-tuning이 적합**: ① 데이터가 충분할 때 ② 도메인이 많이 다를 때(병리 조직, X-ray, 피부 병변처럼 ImageNet과 질감·색이 다른 의료 영상) ③ 조금이라도 높은 성능이 중요하고 GPU 시간이 있을 때.
     - 이번 PathMNIST는 학습 이미지가 89,996장(위 데이터셋 정보 출력)으로 많고 ImageNet과 도메인이 달라서 **일반적으로는 Fine-tuning이 유리한 조건**입니다. 실제로 어느 쪽이 나았는지는 Part 5와 최종 리포트의 자동 생성 표에서 확인합니다.

---

## 📊 Part 5: 성능 비교 및 평가

In [ ]:
# ============================================
# 모델 평가 함수
# ============================================

def evaluate_model(model, test_loader, device):
    """
    테스트 데이터셋에서 모델 평가

    Args:
        model: 평가할 모델
        test_loader: 테스트 데이터 로더
        device: 연산 장치 (cuda/cpu)

    Returns:
        (성능 지표 dict, 예측값 list, 실제값 list)
    """
    model.eval()                                     # 평가 모드
    all_preds = []                                   # 예측 모음
    all_labels = []                                  # 정답 모음

    with torch.no_grad():                            # 기울기 끄기
        for images, labels in test_loader:
            images = images.to(device)
            if len(labels.shape) > 1:                # [N, 1] → [N]
                labels = labels.squeeze().long()

            outputs = model(images)                  # 클래스별 점수
            _, predicted = outputs.max(1)            # 가장 높은 점수의 클래스
            all_preds.extend(predicted.cpu().numpy())  # GPU → CPU → numpy로 모으기
            all_labels.extend(labels.numpy())

    # 성능 지표 계산
    metrics = {
        'accuracy': accuracy_score(all_labels, all_preds),                       # 정확도
        'precision': precision_score(all_labels, all_preds, average='weighted', zero_division=0), # 클래스별 정밀도를 샘플 수로 가중 평균 ([추가] 한 번도 예측 안 된 클래스는 경고 대신 0으로)
        'recall': recall_score(all_labels, all_preds, average='weighted'),       # 클래스별 재현율 가중 평균
        'f1': f1_score(all_labels, all_preds, average='weighted')                # 클래스별 F1 가중 평균
    }

    return metrics, all_preds, all_labels

In [ ]:
# 모든 모델 평가
models_to_evaluate = {
    'Simple CNN': simple_cnn,
    'Feature Extraction': feature_extractor,
    'Fine-tuning': finetuning_model
}

all_results = {}                                     # 모델별 결과 저장

for name, model in models_to_evaluate.items():       # 모델을 하나씩 평가
    metrics, preds, labels = evaluate_model(model, test_loader, device)
    all_results[name] = {
        'metrics': metrics,
        'predictions': preds,
        'labels': labels
    }
    print(f"\n{name}:")
    print(f"  Accuracy: {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall: {metrics['recall']:.4f}")
    print(f"  F1 Score: {metrics['f1']:.4f}")

In [ ]:
# 성능 비교 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))      # 왼쪽 지표 막대, 오른쪽 검증 정확도 곡선

# 1. 성능 지표 비교
metrics_df = pd.DataFrame({
    name: result['metrics'] for name, result in all_results.items()
}).T                                                 # 행 = 모델, 열 = 지표

metrics_df.plot(kind='bar', ax=axes[0], color=['#8FBC8F', '#87CEEB', '#D4C4A8', '#DDA0DD'])  # 모델별 4개 지표 막대
axes[0].set_title('Model Performance Comparison')
axes[0].set_ylabel('Score')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].legend(loc='lower right')
axes[0].set_ylim(0, 1)

# 2. 학습 곡선 비교 (Val Accuracy)
histories = {
    'Simple CNN': simple_history,
    'Feature Extraction': fe_history,
    'Fine-tuning': ft_history
}

for name, history in histories.items():              # 모델별 검증 정확도 곡선
    axes[1].plot(history['val_acc'], label=name)
axes[1].set_title('Validation Accuracy Comparison')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n성능 요약:")
print(metrics_df.round(4))                           # 소수 넷째 자리까지 표로 출력

In [ ]:
# 최고 성능 모델의 혼동 행렬
best_model_name = max(all_results.items(), key=lambda x: x[1]['metrics']['accuracy'])[0]  # 정확도 1위 모델
best_result = all_results[best_model_name]

fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(best_result['labels'], best_result['predictions'])   # [실제 클래스 x 예측 클래스] 개수 표

# CIFAR-10의 경우 클래스가 많으므로 비율로 표시
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]        # 행(실제 클래스)별 합으로 나눠 비율로

sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)              # 대각선 = 클래스별 재현율
ax.set_title(f'Confusion Matrix - {best_model_name}')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---

## 📝 Part 6: 최종 과제

### ✏️ [과제 6-1] 다른 사전학습 모델 시도하기

ResNet18 외에 다른 사전학습 모델(VGG16, EfficientNet 등)을 사용하여 성능을 비교해보세요.

In [ ]:
# [과제 6-1] 다른 사전학습 모델: EfficientNet-B0 (Feature Extraction 방식)
# 왜 EfficientNet-B0? VGG16(약 1.4억 개 파라미터)보다 훨씬 가볍고(약 530만 개) ImageNet 성능은 더 좋아서 Colab에서 돌리기 적당합니다.
# 공정한 비교를 위해 ResNet18 Feature Extraction과 같은 조건(분류기만 학습, lr=0.001, StepLR, 최대 15에폭)으로 학습합니다.

def create_efficientnet_fe(num_classes):
    """EfficientNet-B0 Feature Extraction 모델 생성"""
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)  # ImageNet 사전학습 가중치
    for param in model.parameters():
        param.requires_grad = False                        # 전부 동결 → 특징 추출기로만 사용
    num_features = model.classifier[-1].in_features        # 마지막 Linear의 입력 크기 (1280)
    model.classifier[-1] = nn.Linear(num_features, num_classes)  # 1000개 클래스용 출력층 → 우리 클래스 수로 교체 (이 층만 학습)
    return model

torch.manual_seed(42)                                      # 재현성
efficientnet = create_efficientnet_fe(NUM_CLASSES)         # 모델 생성
eff_trainable = sum(p.numel() for p in efficientnet.parameters() if p.requires_grad)  # 학습 가능 파라미터
eff_total = sum(p.numel() for p in efficientnet.parameters())                         # 전체 파라미터
print(f"EfficientNet-B0 (FE) - Trainable: {eff_trainable:,} / Total: {eff_total:,} ({eff_trainable/eff_total*100:.2f}%)")

criterion = nn.CrossEntropyLoss()                                        # 손실 함수
optimizer = optim.Adam(efficientnet.classifier.parameters(), lr=0.001)   # 분류기만 업데이트
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5) # 5에폭마다 학습률 절반

start_time = time.time()                                   # 시간 측정 시작
efficientnet, eff_history = train_model(
    efficientnet, train_loader, val_loader,
    criterion, optimizer, scheduler,
    num_epochs=15, patience=5, device=device               # ResNet18 FE와 같은 조건
)
train_times['EfficientNet-B0 (FE)'] = time.time() - start_time   # 학습 시간 저장
plot_history(eff_history, "EfficientNet-B0 Feature Extraction")  # 학습 곡선

# 테스트 평가 후 다른 모델들과 같은 결과 사전(all_results)에 추가
metrics, preds, labels = evaluate_model(efficientnet, test_loader, device)
all_results['EfficientNet-B0 (FE)'] = {'metrics': metrics, 'predictions': preds, 'labels': labels}
print("\nEfficientNet-B0 (FE) 테스트 결과:")
for k, v in metrics.items():                               # accuracy, precision, recall, f1
    print(f"  {k}: {v:.4f}")

### ✏️ [과제 6-2] 최종 리포트 작성

---

## CNN/전이학습 실습 리포트

**학습자 이름**: 조태희

**작성 날짜**: 2026-11-09

### 1. 실험 요약

아래 표는 바로 아래 코드 셀이 **이 노트북의 실제 실행 결과(`all_results`, `train_times`)로 자동 생성**합니다. 숫자를 손으로 옮겨 적다가 틀릴 일이 없고, 다시 실행하면 표도 함께 갱신됩니다.

- 데이터: PathMNIST(대장 조직 병리 이미지, 9개 클래스), 28×28 원본을 224×224로 확대해 사용
- 학습 조건: `QUICK_MODE=True`일 때 학습 20,000장 / 검증 4,000장, 테스트는 전체
- 지표: Accuracy와, 클래스별 F1을 클래스 크기로 가중 평균한 weighted F1

In [ ]:
# 📋 실험 요약 표 자동 생성 (실제 실행 결과 사용)
from IPython.display import Markdown, display        # 마크다운 표를 셀 출력으로 보여 주는 도구

trainables = {                                       # 모델별 학습 가능 파라미터 수
    'Simple CNN': sum(p.numel() for p in simple_cnn.parameters() if p.requires_grad),
    'Feature Extraction': sum(p.numel() for p in feature_extractor.parameters() if p.requires_grad),
    'Fine-tuning': sum(p.numel() for p in finetuning_model.parameters() if p.requires_grad),
    'EfficientNet-B0 (FE)': eff_trainable,
}
rows = ["| 모델 | Test Accuracy | Test F1 (weighted) | 학습 시간 | 학습 가능 파라미터 |",
        "|------|:---:|:---:|:---:|:---:|"]           # 표 머리글
for name, res in all_results.items():                # 평가한 모델마다 한 줄
    m = res['metrics']
    minutes = train_times.get(name, float('nan')) / 60   # 초 → 분
    rows.append(f"| {name} | {m['accuracy']:.4f} | {m['f1']:.4f} | {minutes:.1f}분 | {trainables.get(name, 0):,} |")
display(Markdown("\n".join(rows)))                   # 표 출력

best = max(all_results, key=lambda n: all_results[n]['metrics']['accuracy'])   # 정확도 1위
fe_acc = all_results['Feature Extraction']['metrics']['accuracy']
ft_acc = all_results['Fine-tuning']['metrics']['accuracy']
print(f"Test Accuracy 1위: {best} ({all_results[best]['metrics']['accuracy']:.4f})")
print(f"Fine-tuning − Feature Extraction 정확도 차이: {ft_acc - fe_acc:+.4f}")
print(f"실행 조건: QUICK_MODE={QUICK_MODE}, Train {len(train_dataset):,}장 / Val {len(val_dataset):,}장 / "
      f"Test {len(test_dataset):,}장, 장치={device}")

### 2. CNN 구조 분석

- 합성곱 연산의 역할: 3×3 크기의 작은 필터가 이미지를 한 칸씩 훑으며 **국소 패턴**(선, 모서리, 색 경계, 질감)을 찾아 특징맵을 만듭니다. 같은 필터를 이미지의 모든 위치에서 **공유**하므로 파라미터가 적고, 패턴이 어디에 있든 똑같이 찾아냅니다. 층이 깊어질수록 앞 층의 특징을 조합해 세포 모양·조직 구조 같은 **복잡한 특징**을 인식합니다.
- 풀링 연산의 역할: 2×2 칸마다 가장 큰 값만 남겨(MaxPool) 가로·세로를 절반으로 줄입니다(224→112→56→28→14). 계산량과 메모리를 줄이고, 특징이 1~2픽셀 움직여도 결과가 크게 바뀌지 않게 해 주며, 다음 층의 필터가 원본 기준으로 **더 넓은 영역**을 보게 만듭니다.
- 채널 수가 증가하는 이유: 풀링으로 면적이 1/4씩 줄어드는 대신 채널(=찾는 특징의 종류)을 2배씩 늘려(32→64→128→256) **정보량을 유지**합니다. 또 깊은 층일수록 단순한 특징들의 조합 가짓수가 많아지므로 더 많은 종류의 특징맵이 필요합니다.

### 3. 전이학습 분석

- Feature Extraction의 장단점: (장점) 분류기만 학습하므로 학습 가능한 파라미터가 4,617개뿐이라 빠르고 메모리를 적게 쓰며, 데이터가 적어도 과적합 위험이 낮습니다. (단점) ImageNet(일상 사진)에서 배운 특징을 그대로 써야 해서, 병리 조직 이미지처럼 **도메인이 다르면 성능에 한계**가 있습니다.
- Fine-tuning의 장단점: (장점) 마지막 블록(layer4)까지 새 데이터에 맞게 조정하므로 도메인 차이를 줄여 **더 높은 성능**을 기대할 수 있습니다. (단점) 학습할 파라미터가 8,398,345개로 많아 시간과 GPU 메모리가 더 들고, 데이터가 적으면 과적합되거나 사전학습 지식이 망가질 수 있어 **작은 학습률**이 필요합니다.
- 적합한 사용 시나리오: 데이터가 적거나 GPU 시간이 부족하거나 빠른 베이스라인이 필요하면 **Feature Extraction**, 데이터가 충분하고 ImageNet과 많이 다른 의료 영상(병리·X-ray 등)이라면 **Fine-tuning**이 일반적으로 유리합니다. 이번 실험에서 실제로 어느 쪽이 나았는지는 위 자동 생성 표의 `Fine-tuning − Feature Extraction 정확도 차이`로 확인합니다.

### 4. 데이터 증강 효과

- 사용한 데이터 증강 기법: 학습용 변환에만 `RandomHorizontalFlip(p=0.5)`(좌우 반전)과 `RandomRotation(15)`(±15도 회전)을 적용했습니다. 검증·테스트에는 증강 없이 크기 조정과 정규화만 적용했습니다.
- 데이터 증강의 효과: 매 에폭 같은 사진이 조금씩 다른 모습으로 들어가므로 모델이 사진을 '외우기' 어려워지고, 방향·각도가 달라도 같은 조직으로 인식하도록 배워 **과적합을 줄이고 일반화 성능을 높이는** 효과를 기대합니다. 병리 슬라이드는 촬영 방향에 의미가 없어서 반전·회전이 특히 자연스러운 증강입니다. 다만 이번 실습에서는 증강을 끈 모델과 따로 비교하지 않았으므로, 효과를 숫자로 확인하려면 증강 없는 버전을 같은 조건으로 한 번 더 학습해 비교해야 합니다. (학습 정확도는 증강·Dropout이 켜진 상태로 측정되므로 검증 정확도보다 낮게 나올 수 있습니다.)

### 5. 개선 제안

성능을 더 개선하기 위한 방법:

1. **원본 해상도 활용**: 28×28 이미지를 224×224로 늘려도 정보는 늘지 않고 계산량만 약 64배가 됩니다. MedMNIST+의 고해상도 버전(`DataClass(..., size=64)` 또는 128·224)을 쓰거나, Simple CNN은 28×28 그대로 학습하는 편이 효율적입니다.
2. **학습량·학습 범위 늘리기**: `QUICK_MODE=False`로 전체 데이터를 쓰고, Fine-tuning 범위를 `layer3`까지 넓히거나 `CosineAnnealingLR` 같은 스케줄러로 더 오래 학습합니다.
3. **도메인에 맞는 증강과 오답 분석**: 병리 이미지는 염색 농도 차이가 크므로 `ColorJitter`(색조·채도) 증강을 추가하고, 혼동 행렬에서 자주 헷갈리는 클래스 쌍을 찾아 집중 개선합니다.

### 6. 배운 점

이번 과제를 통해 배운 점:

- 가짜 입력으로 특징맵 크기를 직접 추적해 보니, `Conv2d(3×3, padding=1)`은 크기를 유지하고 `MaxPool2d(2)`가 크기를 절반으로 줄인다는 것이 숫자로 확실히 보였습니다.
- 전이학습에서 "어디까지 얼릴지(`requires_grad`)"에 따라 학습할 파라미터가 4,617개 ↔ 8,398,345개로 크게 달라지고, 그만큼 속도·성능·과적합 위험의 균형이 바뀐다는 것을 이해했습니다.
- 코드를 꼼꼼히 읽으며 두 가지 함정을 찾았습니다. `state_dict().copy()`는 얕은 복사라서 최적 모델이 제대로 저장되지 않는다는 점, `plt.imshow`는 '현재 칸'에 그려서 서브플롯마다 `plt.sca(ax)`(또는 `ax.imshow`)가 필요하다는 점입니다.

---

## 🎉 수고하셨습니다!

### 제출 방법
1. 모든 셀을 실행하여 결과가 저장된 상태로 제출
2. 모든 ✏️ 과제 섹션에 답변 작성 완료
3. 최종 리포트 작성 완료

### 평가 기준
- 코드 실행 및 결과 (40%)
- 과제 답변의 충실성 (30%)
- 최종 리포트 분석 품질 (20%)
- 추가 모델 실험 (10%)

## 📌 핵심 정리

- **CNN 블록** = `Conv(특징 찾기) → BatchNorm → ReLU → MaxPool(크기 절반)`. 이 블록을 4번 쌓아 `[3, 224, 224]` → `[256, 14, 14]`로 만들고, **GAP**로 256개 숫자로 요약한 뒤 분류합니다. GAP 덕분에 첫 Linear 층 파라미터가 6,422,656개 → 32,896개로 줄었습니다.
- **전이학습**: Feature Extraction은 분류기만(4,617개), Fine-tuning은 layer4까지(8,398,345개) 학습합니다. Fine-tuning에서는 사전학습 층에 **작은 학습률**, 새 층에 **큰 학습률**을 줍니다.
- **Early Stopping**은 최적 시점의 가중치를 `copy.deepcopy`로 저장해야 진짜로 '최적 모델'이 복원됩니다.
- **결과 숫자는 실행 결과에서 자동으로**: 과적합 진단과 최종 실험 요약 표는 코드가 실제 학습 기록으로 만들어 주므로, 다시 실행해도 답변과 결과가 어긋나지 않습니다.